# SP-4 RL Brain Training (Colab T4 path)

**Primary path is Hetzner cron (`backend/scripts/hetzner_brain_cron.sh` runs nightly at 03:30 UTC).** This notebook is the *manual* alternative — useful for:

1. **First-checkpoint training** before the cron is wired up — produces the v1 checkpoint that Hetzner uses as warm-start.
2. **Ablation experiments** (different hyperparameters, debugging, checkpoint comparison).
3. **Catch-up training** if the Hetzner cron has been failing for several days.

Day-to-day, you should NOT need to run this. The cron handles the nightly cadence.

**Acceptance gate:** `vs_baseline_sharpe_pct >= 10` AND `max_drawdown <= 25%` (spec sec 5.3). The trainer saves the eval JSON regardless; only manually-approved checkpoints get activated.

## 1. Clone repo + install

If the repo is private, paste a fine-scoped GitHub PAT (read access on this repo only). Blank if public.

In [ ]:
import getpass, os, subprocess

PAT = getpass.getpass('GitHub PAT (blank if public): ')
REPO = 'naga1412/v5_Trade_bot'
url = f'https://{PAT + "@" if PAT else ""}github.com/{REPO}.git'

if not os.path.exists('/content/v5_Trade_bot'):
    subprocess.check_call(['git', 'clone', '--depth=1', url, '/content/v5_Trade_bot'])
%cd /content/v5_Trade_bot
!git log -1 --oneline
!pip install --quiet pyarrow requests sqlalchemy aiosqlite asyncpg fakeredis

## 2. Pull the production replay-buffer dump from Hetzner

Trainer needs the closed `shadow_trades` rows. Two options:

* **A. Direct DB connection** (simpler): point the trainer at the production Postgres via SSH tunnel through Cloudflare. Requires `cloudflared access tcp` setup.
* **B. Pre-dumped sqlite snapshot** (recommended for safety): operator runs a small dump script on Hetzner that exports `shadow_trades` + `intermarket_snapshots` to sqlite, scp-uploads it to Drive, and Colab pulls it from Drive. **Read-only path — no risk of training accidentally writing to prod.**

Option B is sketched below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DUMP_PATH = '/content/drive/MyDrive/trading-radar/shadow_trades_dump.sqlite'
import os
if not os.path.exists(DUMP_PATH):
    raise FileNotFoundError(
        f'expected dump at {DUMP_PATH} — run the Hetzner dump script first:\n'
        '  ssh root@95.216.187.204 "docker compose exec -T backend python -m tools.ml.dump_shadow_trades --out /tmp/dump.sqlite"\n'
        '  scp root@95.216.187.204:/tmp/dump.sqlite -> upload to Drive at the path above'
    )

import shutil
shutil.copy(DUMP_PATH, '/content/v5_Trade_bot/data/shadow_dump.sqlite')
print('dump ready.')

## 3. Train the brain on T4 GPU

PPO is small (~25K params). 30 epochs on the dump takes ~10-30 min on T4 (vs ~1-2 hours on Hetzner CPU).

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

from datetime import datetime, timezone
VERSION = 'v1-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUT_DIR = f'data/rl-cache/{VERSION}'
DB_URL = 'sqlite+aiosqlite:///data/shadow_dump.sqlite'

!python -m tools.ml.train_brain \
    --db-url "{DB_URL}" \
    --window-days 365 \
    --out-dir {OUT_DIR} \
    --device auto \
    --version-tag {VERSION} \
    --epochs 30

## 4. Inspect eval JSON

Pretty-print the per-asset metrics + overall Sharpe vs baseline.

In [ ]:
import json, glob
files = sorted(glob.glob(f'{OUT_DIR}/eval_brain_*.json'))
with open(files[-1]) as f:
    doc = json.load(f)

print(f"version: {doc['version']}")
print(f"trained_at: {doc['trained_at']}")
print(f"n_transitions: {doc['train_data_window']['n_transitions']}")
print(f"asset_count: {doc['train_data_window']['asset_count']}")
print(f"epochs_completed: {doc['training']['epochs_completed']}")
print(f"final_entropy: {doc['training']['final_entropy']:.3f}")
print(f"final_policy_loss: {doc['training']['final_policy_loss']:.4f}")
print(f"sha256: {doc['sha256']}")
print()
print('backtest:', doc['backtest_results'])

## 5. Download checkpoint to laptop

Triggers browser download dialog. Save somewhere known, then scp to Hetzner per `tools/ml/README.md`.

In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob(f'{OUT_DIR}/ppo_policy_*.pt')) + sorted(glob.glob(f'{OUT_DIR}/eval_brain_*.json')):
    print('downloading', f)
    files.download(f)